In [1]:
# Sel ini membuat dataset baru untuk Tugas Mandiri Pertemuan 4 dan mengunggahnya ke HDFS
import numpy as np
import pandas as pd

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}
df_tugas4 = pd.DataFrame(data)
df_tugas4.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset dibuat: {df_tugas4.shape[0]} baris")

# Mengunggah ke HDFS
!hdfs dfs -mkdir -p /user/revan18/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /user/revan18/tugas4/
print("Berhasil diunggah ke HDFS: /user/revan18/tugas4/transaksi_september_2026.csv")

Dataset dibuat: 1000 baris
Berhasil diunggah ke HDFS: /user/revan18/tugas4/transaksi_september_2026.csv


In [2]:
#inisiasi Spark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, sum as spark_sum, count, avg

# Membuat SparkSession
spark = SparkSession.builder \
    .appName("Tugas4_Mandiri") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("SparkSession berhasil dibuat!")

26/09/16 22:03:21 WARN Utils: Your hostname, revan18-1701 resolves to a loopback address: 127.0.1.1; using 192.168.1.164 instead (on interface wlp2s0)
26/09/16 22:03:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/16 22:03:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/16 22:03:22 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession berhasil dibuat!


In [3]:
#A. Membaca dan Eksplorasi Awal (Code Cell)
# Membaca data langsung dari HDFS
path_hdfs = "hdfs://localhost:9000/user/revan18/tugas4/transaksi_september_2026.csv"
df = spark.read.csv(path_hdfs, header=True, inferSchema=True)

# Menampilkan struktur, jumlah baris, dan 10 baris pertama
df.printSchema()
print(f"Jumlah baris: {df.count()}")
df.show(10)

root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Jumlah baris: 1000
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|        

In [4]:
#B. Menangani Data Kosong (Code Cell)
# Menghitung jumlah nilai kosong pada kolom rating
jumlah_kosong = df.filter(col("rating").isNull()).count()
print(f"Jumlah baris dengan rating kosong: {jumlah_kosong}")

# Menangani dengan mengisi nilai kosong (fill) menjadi 0
df = df.na.fill({"rating": 0})

Jumlah baris dengan rating kosong: 204


In [5]:
#C. Transformasi Data (Code Cell)
# Menambahkan kolom total_pendapatan
df = df.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))

# Menambahkan kolom tier_transaksi menggunakan fungsi when()
df = df.withColumn("tier_transaksi", 
                   when(col("total_pendapatan") > 500000, "Besar")
                   .otherwise("Kecil"))

df.select("order_id", "total_pendapatan", "tier_transaksi").show(5)

+--------+----------------+--------------+
|order_id|total_pendapatan|tier_transaksi|
+--------+----------------+--------------+
|ORD-3000|          270000|         Kecil|
|ORD-3001|          600000|         Besar|
|ORD-3002|          480000|         Kecil|
|ORD-3003|         2100000|         Besar|
|ORD-3004|          600000|         Besar|
+--------+----------------+--------------+
only showing top 5 rows



In [6]:
#D. Analisis dengan GroupBy (Code Cell)
# 1. Kategori dengan total_pendapatan tertinggi
print("1. Kategori dengan total pendapatan tertinggi:")
df.groupBy("kategori").agg(spark_sum("total_pendapatan").alias("total_pendapatan_kategori")) \
    .orderBy(col("total_pendapatan_kategori").desc()).show(1)

# 2. Kota dengan jumlah transaksi tier "Besar" terbanyak
print("2. Kota dengan jumlah transaksi tier Besar terbanyak:")
df.filter(col("tier_transaksi") == "Besar").groupBy("kota") \
    .agg(count("order_id").alias("jumlah_transaksi_besar")) \
    .orderBy(col("jumlah_transaksi_besar").desc()).show(1)

# 3. Rata-rata rating untuk masing-masing metode_pembayaran
print("3. Rata-rata rating per metode pembayaran:")
df.groupBy("metode_pembayaran").agg(avg("rating").alias("rata_rata_rating")).show()

1. Kategori dengan total pendapatan tertinggi:
+------------+-------------------------+
|    kategori|total_pendapatan_kategori|
+------------+-------------------------+
|Rumah Tangga|                138665000|
+------------+-------------------------+
only showing top 1 row

2. Kota dengan jumlah transaksi tier Besar terbanyak:
+----+----------------------+
|kota|jumlah_transaksi_besar|
+----+----------------------+
|Solo|                    92|
+----+----------------------+
only showing top 1 row

3. Rata-rata rating per metode pembayaran:
+-----------------+------------------+
|metode_pembayaran|  rata_rata_rating|
+-----------------+------------------+
|              COD|3.3745019920318726|
|    Transfer Bank|3.3399209486166006|
|     Kartu Kredit|3.1910569105691056|
|         E-Wallet|             3.292|
+-----------------+------------------+



In [7]:
#E. Menyimpan Hasil ke HDFS (Code Cell)
# Menyimpan kembali ke HDFS dalam format CSV
output_path = "hdfs://localhost:9000/user/revan18/tugas4/hasil_transaksi_olahan"

# mode="overwrite" agar menimpa file jika sebelumnya sudah ada
df.write.csv(output_path, header=True, mode="overwrite")
print("Data berhasil disimpan ke HDFS!")

# Menutup sesi Spark di akhir pengerjaan
spark.stop()

Data berhasil disimpan ke HDFS!
